# PerGAZE ChenLSTMISP ? Kaggle DataParallel (2 GPUs)

In Kaggle Settings select **GPU T4 x2**, enable **Internet**, and add the PerGAZE dataset.
Train uses `train.json`; validation uses `test_seen.json`. The notebook does not use
`support.json` or `test_unseen.json`. The public repository needs no token.
Outputs are written to `/kaggle/working/pergaze_chenlstm_run`.


In [ ]:
import subprocess
import sys
from pathlib import Path

CODE = Path("/kaggle/working/IndividualScanpath")
if (CODE / ".git").is_dir():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ntnghia06/IndividualScanpath.git", str(CODE)], check=True)
SRC = CODE / "PerGAZE/ChenLSTMISP/src"
assert (SRC / "parallel.py").is_file()
assert (SRC / "training_settings.py").is_file(), "Update the code: DataParallel support is missing"


In [ ]:
%pip install -q "numpy>=1.24" "Pillow>=9.1" "scipy>=1.10" "tqdm>=4.65" multimatch-gaze


In [ ]:
import torch
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Enable GPU in Kaggle Settings"
assert torch.cuda.device_count() >= 2, "Select GPU T4 x2"
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

DATA = Path("/kaggle/input/datasets/ngtrongnghia/pergaze/dataset")
RUN = Path("/kaggle/working/pergaze_chenlstm_run")
for name in ("train.json", "test_seen.json", "images", "attention_reasoning"):
    assert (DATA / name).exists(), f"Missing: {DATA / name}"

COMMON = [
    "--train_file", str(DATA / "train.json"),
    "--val_file", str(DATA / "test_seen.json"),
    "--img_dir", str(DATA / "images"),
    "--att_dir", str(DATA / "attention_reasoning"),
    "--device", "cuda:0", "--gpu_ids", "0", "1",
]
def run_script(script, *arguments):
    subprocess.run([sys.executable, "-u", str(SRC / script), *COMMON, *arguments], check=True)


Validate every record and asset before training. This does not update model weights.


In [ ]:
run_script("preprocess/validate_dataset.py", "--log_root", str(RUN))


Smoke check: duplicated real samples per condition create a batch large enough to exercise both GPUs. Random weights, 3 fixations; not a trained model.


In [ ]:
run_script("train.py", "--smoke_test", "--batch", "4", "--max_length", "3",
           "--rl_sample_number", "2", "--log_root", "/kaggle/working/pergaze_chenlstm_smoke")


Full training uses AiR run defaults directly: 40 epochs, RL from epoch index 20,
global batch 2 (one sample per GPU), validation batch 1. There is no selector.
Run this cell for a new run; use resume below if a checkpoint exists.
The first run downloads pretrained ImageNet ResNet50 weights.


In [ ]:
run_script("train.py", "--log_root", str(RUN), "--workers", "2",
           )


Evaluate best.pth on the validation file. This is validation evaluation, not an independent unseen test.


In [ ]:
run_script("test.py", "--log_root", str(RUN), "--workers", "2")


Optional resume: run this cell instead of the full-training cell. Restore the run directory first if starting a new Kaggle session. Resume restores saved training settings and schedule. An explicit --epoch may extend the total target.


In [ ]:
# Uncomment to resume:
# run_script("train.py", "--resume", "--log_root", str(RUN),
#            "--workers", "2")


In [ ]:
import json
if (RUN / "evaluation_validation.json").exists():
    result = json.loads((RUN / "evaluation_validation.json").read_text())
    print(json.dumps(result["overall"], indent=2))


Save notebook outputs and checkpoints before ending the session.


In [ ]:
import shutil
output = shutil.make_archive("/kaggle/working/pergaze_chenlstm_results", "zip",
                             root_dir=RUN.parent, base_dir=RUN.name)
print(output)
